# Assignment 3 – QSAR
## Part 1: Exploratory Data Analysis (EDA)

### Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style='ticks')

df = pd.read_csv("bioactivity_preprocessed_data.csv")
print("Original shape:", df.shape)

### Data Cleaning

In [ ]:
df = df.dropna(subset=[
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value"
])
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["standard_value"])
df = df[df["bioactivity_class"] != "intermediate"]
print("After cleaning:", df.shape)

### Deduplication and pIC50 Calculation

In [ ]:
df_clean = (
    df
    .groupby("canonical_smiles", as_index=False)
    .agg({
        "molecule_chembl_id": "first",
        "standard_value": "median",
        "bioactivity_class": "first"
    })
)
print("Final dataset size:", df_clean.shape)

df_clean["standard_value"] = df_clean["standard_value"].clip(lower=1e-9)
df_clean["pIC50"] = -np.log10(df_clean["standard_value"] * 1e-9)
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna(subset=["pIC50"])

threshold = 6
df_clean["bioactivity_class"] = np.where(
    df_clean["pIC50"] >= threshold, "active", "inactive"
)

df_clean["pIC50"].describe()

### pIC50 Distribution

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(df_clean["pIC50"], bins=30, color='skyblue', edgecolor='black')
plt.xlabel("pIC50")
plt.ylabel("Count")
plt.title("pIC50 Distribution")
plt.savefig("histogram_pIC50.pdf")
plt.show()

### Active vs Inactive Class Counts

In [ ]:
plt.figure(figsize=(5.5, 5.5))
sns.countplot(x="bioactivity_class", data=df_clean, edgecolor="black")
plt.xlabel("Bioactivity Class")
plt.ylabel("Frequency")
plt.title("Active vs Inactive Counts")
plt.savefig("barplot_bioactivity_class.pdf")
plt.show()

### Save Cleaned Data for Next Steps

In [ ]:
df_clean.to_csv("df_clean.csv", index=False)
print("Saved df_clean.csv")